# Étape 1 — Test de la synthèse vocale éwé (Meta MMS)

**Projet ɖevi ƒe agbalẽ · Hack4Children 2026**

But : décider **ce soir** si `facebook/mms-tts-ewe` est assez bon pour les messages aux familles.

Ce notebook :
1. installe et charge le modèle ;
2. vérifie que les lettres éwé (ɖ, ƒ, ɣ, ŋ, ɔ, ɛ, ʋ, accents) sont bien lues ;
3. génère quelques phrases et des prénoms, à écouter ;
4. compare deux vitesses de parole ;
5. télécharge tous les fichiers audio dans un .zip.

> Dans Colab : *Exécution > Modifier le type d'exécution > GPU T4* (facultatif, le processeur suffit).

In [ ]:
# 1. Installation (torch est déjà présent dans Colab)
!pip install -q transformers scipy

In [ ]:
# 2. Chargement du modèle éwé
import torch, unicodedata, os
from transformers import VitsModel, AutoTokenizer, set_seed

MODELE = "facebook/mms-tts-ewe"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODELE)
model = VitsModel.from_pretrained(MODELE).to(device).eval()
SR = model.config.sampling_rate

print("Appareil :", device, "| fréquence :", SR, "Hz")
print("Romanisation (uroman) nécessaire :", getattr(tokenizer, "is_uroman", False))

## 3. Vérifier l'alphabet

Le modèle **supprime en silence** tout caractère absent de son vocabulaire.
Si une lettre éwé manque, un son disparaît sans message d'erreur. On vérifie donc avant tout.

In [ ]:
vocab = set(tokenizer.get_vocab().keys())
print("Caractères connus du modèle :")
print(" ".join(sorted(c for c in vocab if len(c) == 1)))

def caracteres_inconnus(texte, forme="NFC"):
    t = unicodedata.normalize(forme, texte.lower())
    return sorted({c for c in t if c not in vocab and not c.isspace()})

LETTRES_EWE = "ɖ ƒ ɣ ŋ ɔ ɛ ʋ é è ê à á ã ẽ õ ò ó ù ú ì í"
for forme in ("NFC", "NFD"):
    manquants = caracteres_inconnus(LETTRES_EWE, forme)
    print(f"\nForme {forme} - caractères NON reconnus :",
          [f"{c} (U+{ord(c):04X})" for c in manquants] or "aucun")

**Résultat du 16 nov. :** toutes les lettres éwé sont reconnues. Seuls ê (NFC) et õ (NFC) posent problème : la cellule suivante les corrige.

In [ ]:
# Règle retenue après la vérification de l'alphabet :
# - forme NFC (lettres accentuées composées, comme dans le vocabulaire du modèle)
# - õ n'existe pas composé -> o + tilde combinant (connu du modèle)
# - ê : le circonflexe est inconnu -> remplacé par e (ton montant-descendant perdu, cas rare)
REMPLACEMENTS = {"õ": "o\u0303", "ê": "e"}

def normaliser(texte):
    t = unicodedata.normalize("NFC", texte.lower())
    for avant, apres in REMPLACEMENTS.items():
        t = t.replace(avant, apres)
    return t

def caracteres_restants(texte):
    """Caractères qui seront supprimés en silence par le modèle."""
    return sorted({c for c in normaliser(texte) if c not in vocab and not c.isspace()})

def texte_vu_par_le_modele(texte):
    ids = tokenizer(normaliser(texte))["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    return "".join(x for x in toks if x != tokenizer.pad_token)

print(texte_vu_par_le_modele("Ɖevi ƒe agbalẽ, Kodzo, õ, ê, 15 ŋkeke."))

**Attention, ce que le modèle ne lit pas :**
- **pas de chiffres** → écrire les nombres en toutes lettres (« eve », pas « 2 ») ;
- **pas de c, j, q** → écrire les prénoms en orthographe éwé (**Kodzo** et non Kodjo, **Kosi** et non Kossi) ;
- **pas de ponctuation** (virgule, point) → pas de pause automatique ; les pauses viendront du découpage en segments à l'étape 4.

## 4. Phrases de test

**Jérôme : remplace les phrases éwé** par tes propres traductions (garde l'orthographe complète, avec tons si tu les écris).
Le français est là pour mémoire. Les phrases ci-dessous ne sont que des exemples de départ.

In [ ]:
PHRASES = {
    "bienvenue": {
        "fr": "Bienvenue. Votre enfant est maintenant inscrit dans le carnet ɖevi ƒe agbalẽ.",
        "ewe": "Woezɔ! Ŋdi na wò.",   # À REMPLACER par ta traduction complète
    },
    "rappel_vaccin": {
        "fr": "Après-demain, votre enfant doit recevoir son vaccin au centre de santé. Ce vaccin est gratuit.",
        "ewe": "",                      # À ÉCRIRE
    },
    "rendez_vous_manque": {
        "fr": "Votre enfant n'est pas venu à son rendez-vous de vaccin. Appuyez sur 1 si vous viendrez, sur 2 si vous avez un empêchement.",
        "ewe": "",                      # À ÉCRIRE
    },
}

for cle, p in PHRASES.items():
    if p["ewe"]:
        inc = caracteres_restants(p["ewe"])
        print(f"{cle:20s} | supprimés : {inc or 'aucun'}")
        print(f"{'':20s} | vu par le modèle : {texte_vu_par_le_modele(p['ewe'])}")

In [ ]:
# 5. Génération et écoute
import numpy as np, scipy.io.wavfile as wavfile
from IPython.display import Audio, display

SORTIE = "/content/voix_test"
os.makedirs(SORTIE, exist_ok=True)

def synthese(texte, vitesse=1.0, graine=42):
    """Renvoie le signal audio (numpy) pour un texte éwé."""
    set_seed(graine)                 # même graine = même voix à chaque fois
    model.speaking_rate = vitesse    # < 1 : plus lent, > 1 : plus rapide
    entree = tokenizer(normaliser(texte), return_tensors="pt").to(device)
    with torch.no_grad():
        onde = model(**entree).waveform[0].cpu().numpy()
    return onde

def sauver(onde, nom):
    chemin = f"{SORTIE}/{nom}.wav"
    wavfile.write(chemin, SR, (onde * 32767).astype(np.int16))
    return chemin

for cle, p in PHRASES.items():
    if not p["ewe"]:
        print(f"[{cle}] pas encore de texte éwé - ignoré")
        continue
    onde = synthese(p["ewe"])
    sauver(onde, cle)
    print(f"[{cle}] {len(onde)/SR:.1f} s - {p['fr']}")
    display(Audio(onde, rate=SR))

## 6. Vitesse de parole

Pour des mères qui écoutent au téléphone, un débit un peu plus lent peut être plus clair. Compare 0,85 et 1,0.

In [ ]:
phrase_test = next((p["ewe"] for p in PHRASES.values() if p["ewe"]), "")
for v in (0.85, 1.0):
    onde = synthese(phrase_test, vitesse=v)
    sauver(onde, f"vitesse_{v}")
    print(f"Vitesse {v}")
    display(Audio(onde, rate=SR))

## 7. Prénoms

Les prénoms seront générés à la volée dans les messages. Ils doivent être écrits en orthographe éwé (pas de c, j, q).

In [ ]:
PRENOMS = ["Kosi", "Ama", "Yao", "Afi", "Kodzo", "Akosiwa", "Komi", "Abla", "Mawuli", "Sena"]   # orthographe éwé
for p in PRENOMS:
    if caracteres_restants(p):
        print("Attention", p, ":", caracteres_restants(p))

for prenom in PRENOMS:
    onde = synthese(prenom, vitesse=0.9)
    sauver(onde, f"prenom_{prenom}")
    print(prenom)
    display(Audio(onde, rate=SR))

## 8. (Facultatif) Essai rapide en kabiyè

Pour la finale. Laisse vide si personne ne peut juger la prononciation pour l'instant.

In [ ]:
PHRASE_KABIYE = ""   # une phrase kabiyè à tester

if PHRASE_KABIYE:
    tok_kbp = AutoTokenizer.from_pretrained("facebook/mms-tts-kbp")
    mod_kbp = VitsModel.from_pretrained("facebook/mms-tts-kbp").to(device).eval()
    set_seed(42)
    with torch.no_grad():
        onde = mod_kbp(**tok_kbp(PHRASE_KABIYE, return_tensors="pt").to(device)).waveform[0].cpu().numpy()
    sauver(onde, "kabiye_test")
    display(Audio(onde, rate=mod_kbp.config.sampling_rate))

In [ ]:
# 9. Télécharger tous les fichiers générés (.zip)
import shutil
from google.colab import files
archive = shutil.make_archive("/content/voix_test_mms_ewe", "zip", SORTIE)
files.download(archive)

## 10. Grille de décision (à remplir après écoute)

| Critère | Note /5 | Remarque |
|---|---|---|
| Compréhensible du premier coup | | |
| Tons et voyelles corrects (ɔ, ɛ, nasales) | | |
| Prénoms bien prononcés | | |
| Voix naturelle, pas trop robotique | | |
| Vitesse retenue (0,85 ou 1,0) | | |

**Décision :**
- **≥ 4 partout** → MMS pour toutes les phrases ; tu réenregistres seulement les ratés.
- **3** → MMS pour prénoms, vaccins et centres ; ta voix pour les phrases fixes.
- **≤ 2** → ta voix pour tout ; MMS garde seulement les prénoms.